# Classification model - hyperParameter Tuning 

Uses RandomSearch tuner.

### Tunes:

1. Number of hidden layers (1–5).

2. Number of units per layer (32–512).

3. Dropout usage & rate.

4. Learning rate.

### 1. Prepare the Data

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler


df=pd.read_csv('../data/Churn_Modelling.csv')

In [9]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


### Geography and Gender are categorical (text).

Machine learning models need numerical input, so we use LabelEncoder to convert them:

Example: France=0, Spain=1, Germany=2.

Male=1, Female=0.

In [10]:
# Drop irrelevant columns
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode categorical variables
le_geo = LabelEncoder()
df["Geography"] = le_geo.fit_transform(df["Geography"])

le_gender = LabelEncoder()
df["Gender"] = le_gender.fit_transform(df["Gender"])

# Features and target
X = df.drop("Exited", axis=1) # Features
y = df["Exited"] # Target variable


## Scale features
# Features (e.g., balance, salary, age) have different scales.
## StandardScaler normalizes them → mean = 0, variance = 1.
# Neural networks train faster and more stably when features are scaled.
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
# Split data: 80% for training, 20% for validation.
# Validation set helps measure how well the model generalizes.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [11]:
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,0,0,42,2,0.00,1,1,1,101348.88,1
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1
3,699,0,0,39,1,0.00,2,0,0,93826.63,0
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0


### 2. Define the Model - with keras Tuner (hyperparameter tuning)

hp.Int() defines an integer hyperparameter.

min_value=32 → the smallest number of neurons tuner will try.

max_value=512 → the largest number of neurons tuner will try.

step=32 → tuner increases the value in increments of 32.

In [15]:
import keras
from keras import layers
from keras_tuner import RandomSearch

input_dim = X_train.shape[1]  # number of features
num_classes = 2  # binary classification (Exited: 0 or 1)

def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    # Hidden layers (tuned number + neurons)
    for i in range(hp.Int("num_layers", 1, 5)):
        model.add(layers.Dense(
            units=hp.Int(f"units_{i}", min_value=32, max_value=512, step=32),
            activation="relu"
        ))
        if hp.Boolean(f"dropout_{i}"):
            model.add(layers.Dropout(rate=hp.Float(f"dropout_rate_{i}", 0.1, 0.5, step=0.1)))

    # Output layer (binary classification)
    model.add(layers.Dense(1, activation="sigmoid"))

    # Compile
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


### 3. Define and run the tuner

In [16]:
tuner = RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=10,              # number of hyperparam configs to try
    executions_per_trial=2,     # average performance over runs
    directory="project",
    project_name="Customer_Churn"
)

# Run search
tuner.search(X_train, y_train, epochs=20, validation_data=(X_val, y_val))

Trial 10 Complete [00h 00m 59s]
val_accuracy: 0.8197500109672546

Best val_accuracy So Far: 0.8230000138282776
Total elapsed time: 00h 16m 08s


### 4. Get the best model

In [17]:
# Best hyperparameters summary
tuner.results_summary()

# Get best model
best_model = tuner.get_best_models(num_models=1)[0]

# Evaluate on validation set
val_loss, val_acc = best_model.evaluate(X_val, y_val)
print("Best Validation Accuracy:", val_acc)


Results summary
Results in project/Customer_Churn
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 03 summary
Hyperparameters:
num_layers: 2
units_0: 160
dropout_0: False
learning_rate: 0.001
units_1: 448
dropout_1: True
units_2: 32
dropout_2: False
units_3: 96
dropout_3: False
units_4: 288
dropout_4: True
dropout_rate_0: 0.5
dropout_rate_1: 0.4
dropout_rate_2: 0.5
dropout_rate_3: 0.2
Score: 0.8230000138282776

Trial 00 summary
Hyperparameters:
num_layers: 1
units_0: 32
dropout_0: False
learning_rate: 0.01
Score: 0.8222499787807465

Trial 09 summary
Hyperparameters:
num_layers: 1
units_0: 256
dropout_0: False
learning_rate: 0.01
units_1: 128
dropout_1: True
units_2: 32
dropout_2: False
units_3: 128
dropout_3: True
units_4: 96
dropout_4: True
dropout_rate_0: 0.30000000000000004
dropout_rate_1: 0.2
dropout_rate_2: 0.2
dropout_rate_3: 0.2
dropout_rate_4: 0.5
Score: 0.8197500109672546

Trial 07 summary
Hyperparameters:
num_layers: 2
units_0: 256
dropout_0: True

/Users/baqirrizvi/Desktop/ai/DeepLearning/deeplearning/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8240 - loss: 0.4421
Best Validation Accuracy: 0.8240000009536743


In [18]:
print("Best Validation Accuracy:", val_acc)

Best Validation Accuracy: 0.8240000009536743
